# Added-column drift: orders / CSV

This notebook processes only its own dataset. Run independently in cell order. Shared logic lives in schema_drift.py.

## Environment
Use the configured Linux Spark/Hadoop environment and a PySpark kernel. Start Jupyter in this directory. A compatible Iceberg runtime must be on the classpath or configured through ICEBERG_SPARK_RUNTIME_JAR. Default paths use project-relative file URIs; shared clusters require shared warehouse, landing and quarantine paths. Use a fresh kernel to change Spark settings.

In [ ]:
NOTEBOOK_NAME = "schema_drifting_csv_orders.ipynb"
import os
from pathlib import Path
from pyspark.sql import SparkSession, types as T, functions as F

CATALOG = "datalake"
WAREHOUSE = os.environ.get(
    "SCHEMA_DRIFT_WAREHOUSE", (Path.cwd() / "warehouse").as_uri()
)
DATA_ROOT = os.environ.get("SCHEMA_DRIFT_DATA_ROOT", (Path.cwd() / "datasets").as_uri())
BRONZE_ROOT = os.environ.get(
    "SCHEMA_DRIFT_BRONZE_ROOT", (Path.cwd() / "bronze").as_uri()
)
QUARANTINE_ROOT = os.environ.get(
    "SCHEMA_DRIFT_QUARANTINE_ROOT", (Path.cwd() / "quarantine").as_uri()
)

builder = (
    SparkSession.builder.appName(NOTEBOOK_NAME)
    .master(os.environ.get("SPARK_MASTER", "local[2]"))
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.datalake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.datalake.type", "hadoop")
    .config("spark.sql.catalog.datalake.warehouse", WAREHOUSE)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.caseSensitive", "true")
    .config("spark.sql.shuffle.partitions", "2")
)
if os.environ.get("ICEBERG_SPARK_RUNTIME_JAR"):
    builder = builder.config("spark.jars", os.environ["ICEBERG_SPARK_RUNTIME_JAR"])
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## Dataset and per-file drift policy
One CSV adds sales_channel; another has no extra field. Each file is inferred independently so NEW_COLUMN is attributed only to the file that adds it. Extra columns are audited and dropped. Invalid order_id is quarantined; valid fields are cast to the target contract. CSV inference may also report amount as double versus target decimal. A header-only file still receives its own audit.

In [ ]:
from schema_drift import process_csv_batch, read_output, BAD_SCHEMA

DATASET_NAME = "orders"
SOURCE_PATH = DATA_ROOT.rstrip("/") + "/orders/added_column"

expected_schema = T.StructType(
    [
        T.StructField("order_id", T.IntegerType(), False),
        T.StructField("customer_id", T.IntegerType(), False),
        T.StructField("amount", T.DecimalType(12, 2), True),
    ]
)

config = dict(
    dataset_name=DATASET_NAME,
    source_path=SOURCE_PATH,
    audit_catalog=CATALOG,
    bronze_path=BRONZE_ROOT.rstrip("/") + "/" + DATASET_NAME,
    expected_schema=expected_schema,
    notebook_name=NOTEBOOK_NAME,
    quarantine_path=QUARANTINE_ROOT.rstrip("/") + "/" + DATASET_NAME,
    cache_enabled=True,
)  # Set False to disable input/split caching.


## Process the landing batch
Inventory files first, including empty files. Each file keeps its own schema and full path, but all share one batch_id. Input and correct/bad splits are cached with MEMORY_AND_DISK by default so counts and writes reuse them. Set cache_enabled=False to disable caching. Each file releases its caches in a finally block on success or failure. Bronze and quarantine are file directories in the same format as landing: CSV to CSV, JSONL to JSONL, Parquet to Parquet. Only audit metadata is stored in an Iceberg table backed by Parquet. Correct records use the provided expected schema; quarantine retains original values in source_record_json plus error_columns and error_reasons. CSV encodes those arrays as JSON strings within CSV cells. _batch_id, _run_id and _source_file provide write traceability. Unreadable files are recorded with processing_error and unknown counts, and processing continues to the next file.

In [ ]:
result = process_csv_batch(spark, **config)

summary = result["summary"]
file_audits = result["file_audits"]

correct_records = result["correct_records"]
bad_records = result["bad_records"]
print(summary)
# Display only a sample. Spark performs the full processing/counts on executors.
correct_records.show(20, truncate=False)
bad_records.show(20, truncate=False)


## Verify persisted counts in the same file audit
reconcile_file reads Bronze and quarantine back using batch/run/file metadata. RECONCILED means source = good + bad and both destination counts match. RECONCILED does not mean every source record was good; successful quarantine also reconciles. FAILED or NOT_VERIFIABLE never claims a successful count check. The notebook verifies the actual destinations, not just input-derived counts.

In [ ]:
assert summary["file_count"] == 3
assert summary["good_record_count"] == 4
assert summary["bad_record_count"] == 1
assert summary["reconciliation_status"] == "RECONCILED"
assert summary["record_count"] == 5
assert summary["bronze_written_count"] == 4
assert summary["quarantine_written_count"] == 1

file_audit_df = spark.table("datalake.operations.schema_drift").where(
    F.col("batch_id") == summary["batch_id"]
)
assert file_audit_df.count() == 3
assert (
    file_audit_df.groupBy("source_file_path").count().where("count != 1").count() == 0
)
file_audit_df.select(
    "source_file_name",
    "drift_types",
    "record_count",
    "good_record_count",
    "bad_record_count",
    "bronze_written_count",
    "quarantine_written_count",
    "reconciliation_status",
    "action_taken",
    "status",
).orderBy("source_file_name").show(truncate=False)
# Inspect persisted destinations, scoped to this batch (re-runs cannot pollute the display).
for file_audit in file_audits:
    if file_audit["good_record_count"]:
        read_output(
            spark,
            file_audit["bronze_file_path"],
            file_audit["output_format"],
            correct_records.schema,
        ).show(20, truncate=False)
    if file_audit["bad_record_count"]:
        read_output(
            spark,
            file_audit["quarantine_file_path"],
            file_audit["output_format"],
            BAD_SCHEMA,
        ).show(20, truncate=False)
bad_records.select(F.explode("error_reasons").alias("reason")).groupBy(
    "reason"
).count().show(truncate=False)


## File audit writer and batch summary
write_audit_to_iceberg writes one dictionary per file to datalake.operations.schema_drift, including schema and reconciliation results. The processors call it automatically; do not append a second time. The session view schema_drift_batch_summary aggregates the persisted file rows and preserves unknown counts. Hadoop catalog view support varies, so the reusable helper registers a temporary view; batch_summary_query also supplies SQL for a catalog-supported permanent view.

In [ ]:
spark.table("schema_drift_batch_summary").where(
    F.col("batch_id") == summary["batch_id"]
).show(truncate=False)
# Do not COUNT file audit rows as batches. Group by batch_id or query the summary view.
spark.sql("""SELECT dataset_name, status, COUNT(*) AS batches
FROM schema_drift_batch_summary GROUP BY dataset_name, status""").show()


## Examples and limitations
Compare per-file added_columns. Explain why amount may also produce TYPE_CHANGED. Add an extra column to a new file and check that the other file audit is unchanged.

Replay protection rejects an explicitly reused audited batch ID; automatically generated IDs do not deduplicate the same source files. Landing files must remain immutable: inference, cache eviction or disabling caching can require source rereads. Returned split DataFrames are uncached after processing; use persisted destinations for subsequent inspection. Bronze, quarantine and audit commits remain separate: persisted counts detect mismatches but do not provide atomic recovery. File-inventory/audit-table setup errors abort before file processing. Audit-write failure stops the batch. This is schema drift, not distribution drift. Per-file processing favors clarity over high-throughput optimization; production can group compatible files while retaining file-level metadata.